Populate the `ENZYME_SUBSTRATE_RELATIONSHIP` table with information from PSP

In [1]:
from pathlib import Path
import pandas as pd
import sqlite3

In [2]:
conn = sqlite3.connect('../sqlite_backend.db')

In [3]:
phosphositeplus_kin_sub_filepath = Path('../resources/Kinase_Substrate_Dataset_20231219')
kin_sub_df = pd.read_csv(phosphositeplus_kin_sub_filepath, sep='\t', skiprows=3)
kin_sub_df

,GENE,KINASE,KIN_ACC_ID,KIN_ORGANISM,SUBSTRATE,SUB_GENE_ID,SUB_ACC_ID,SUB_GENE,SUB_ORGANISM,SUB_MOD_RSD,SITE_GRP_ID,SITE_+/-7_AA,DOMAIN,IN_VIVO_RXN,IN_VITRO_RXN,CST_CAT#
0,Dyrk2,DYRK2,Q5U4C9,mouse,NDEL1,83431.0,Q9ERR1,Ndel1,mouse,S336,1869686801,LGSsRPSsAPGMLPL,NaN,,X,NaN
1,Pak2,PAK2,Q64303,rat,MEK1,170851.0,Q01986,Map2k1,rat,S298,448284,RtPGRPLsSYGMDSR,Pkinase,,X,9128; 98195
2,Pak2,PAK2,Q64303,rat,PRKD1,85421.0,Q9WTQ1,Prkd1,rat,S203,449896,GVRRRRLsNVsLTGL,NaN,X,,NaN
3,Pak2,PAK2,Q64303,rat,prolactin,24683.0,P01237,Prl,rat,S206,451732,IRCLRRDsHKVDNYL,Hormone_1,,X,NaN
4,Pak2,PAK2,Q64303,rat,prolactin,5617.0,P01236,PRL,human,S207,451732,LHCLRRDsHKIDNYL,Hormone_1,,X,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23222,ULK2,ULK2,Q8IYT8,human,Raptor,57521.0,Q8N122,RPTOR,human,S855,3205935,QRVLDtssLtQsAPA,NaN,X,,NaN
23223,ULK2,ULK2,Q8IYT8,human,Raptor,57521.0,Q8N122,RPTOR,human,S859,2024885,DtssLtQsAPAsPtN,NaN,X,,NaN
23224,ULK2,ULK2,Q8IYT8,human,SEC16A,9919.0,O15027,SEC16A,human,S846,55578720,LAQPINFsVSLSNSH,NaN,X,,NaN
23225,ULK2,ULK2,Q8IYT8,human,PIK3C3,5289.0,Q8NEB9,PIK3C3,human,S249,35483209,ESsPILTsFELVKVP,NaN,X,,13857


In [5]:
organism = 'human'
kin_sub_df = kin_sub_df[(kin_sub_df['KIN_ORGANISM'] == organism) & (kin_sub_df['SUB_ORGANISM'] == organism)]
kin_sub_df

,GENE,KINASE,KIN_ACC_ID,KIN_ORGANISM,SUBSTRATE,SUB_GENE_ID,SUB_ACC_ID,SUB_GENE,SUB_ORGANISM,SUB_MOD_RSD,SITE_GRP_ID,SITE_+/-7_AA,DOMAIN,IN_VIVO_RXN,IN_VITRO_RXN,CST_CAT#
7,EIF2AK1,HRI,Q9BQI3,human,eIF2-alpha,1965.0,P05198,EIF2S1,human,S52,447635,MILLsELsRRRIRsI,S1,,X,3597; 9721; 3398; 5199; 53085; 95797
8,EIF2AK1,HRI,Q9BQI3,human,eIF2-alpha,1965.0,P05198,EIF2S1,human,S49,450210,IEGMILLsELsRRRI,S1,,X,NaN
11,PRKCD,PKCD,Q05655,human,HDAC5,10014.0,Q9UQL6,HDAC5,human,S259,447995,FPLRkTAsEPNLKVR,NaN,,X,3443
12,PRKCD,PKCD,Q05655,human,PTPRA iso6,5786.0,P18433-2,PTPRA,human,S204,447612,PLLARSPsTNRKYPP,NaN,X,,NaN
13,PRKCD,PKCD,Q05655,human,Bcl-2,596.0,P10415,BCL2,human,S70,448395,RDPVARtsPLQtPAA,NaN,X,,2834; 2827
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23222,ULK2,ULK2,Q8IYT8,human,Raptor,57521.0,Q8N122,RPTOR,human,S855,3205935,QRVLDtssLtQsAPA,NaN,X,,NaN
23223,ULK2,ULK2,Q8IYT8,human,Raptor,57521.0,Q8N122,RPTOR,human,S859,2024885,DtssLtQsAPAsPtN,NaN,X,,NaN
23224,ULK2,ULK2,Q8IYT8,human,SEC16A,9919.0,O15027,SEC16A,human,S846,55578720,LAQPINFsVSLSNSH,NaN,X,,NaN
23225,ULK2,ULK2,Q8IYT8,human,PIK3C3,5289.0,Q8NEB9,PIK3C3,human,S249,35483209,ESsPILTsFELVKVP,NaN,X,,13857


Need to retrieve the Protein ID of every kinase and the modification id of every substrate

In [14]:
kin_sub_df['SITE_IDENTIFIER'] = kin_sub_df.apply(
    lambda row: f"{row['SUB_ACC_ID']}_{row['SUB_MOD_RSD']}", axis=1)

In [15]:
kin_sub_df[['KIN_ACC_ID', 'SITE_IDENTIFIER']]

,KIN_ACC_ID,SITE_IDENTIFIER
7,Q9BQI3,P05198_S52
8,Q9BQI3,P05198_S49
11,Q05655,Q9UQL6_S259
12,Q05655,P18433-2_S204
13,Q05655,P10415_S70
...,...,...
23222,Q8IYT8,Q8N122_S855
23223,Q8IYT8,Q8N122_S859
23224,Q8IYT8,O15027_S846
23225,Q8IYT8,Q8NEB9_S249


In [17]:
protein_df = pd.read_sql('SELECT PROTEIN_ID, UNIPROT_ACC FROM PROTEIN', conn) 
protein_df

,PROTEIN_ID,UNIPROT_ACC
0,1,A0A087X0M5
1,2,A6NEH6
2,3,A6NIH7
3,4,A6NJR5
4,5,A6NKF7
...,...,...
105714,105715,A0A0D9SG52
105715,105716,A0A1W2PRQ8
105716,105717,C9J4A7
105717,105718,G3V3Y1


In [19]:
modified_site_df = pd.read_sql('SELECT MODIFIED_SITE_ID, SITE_IDENTIFIER FROM MODIFIED_SITE', conn) 
modified_site_df

,MODIFIED_SITE_ID,SITE_IDENTIFIER
0,1,A0A087X0M5_T3
1,2,A0A087X0M5_S19
2,3,A0A087X0M5_S43
3,4,A0A087X0M5_S49
4,5,A0A087X0M5_Y52
...,...,...
7316638,7316639,H0Y8G0_T101
7316639,7316640,H0Y8G0_Y112
7316640,7316641,H0Y8G0_S119
7316641,7316642,H0Y8G0_T120


In [29]:
enzyme_class_df = pd.read_sql('SELECT * FROM ENZYME_CLASS', conn) 
enzyme_class_df

,ENZYME_CLASS_ID,NAME
0,1,Kinase


In [31]:
enzyme_class_id = enzyme_class_df.loc[0]['ENZYME_CLASS_ID']
enzyme_class_id

np.int64(1)

In [48]:
kin_sub_df_mapped = kin_sub_df[['KIN_ACC_ID', 'SITE_IDENTIFIER']].merge(protein_df,
                                                    left_on='KIN_ACC_ID',right_on='UNIPROT_ACC',
                                                    how='inner'
                                                   ).merge(modified_site_df,
                                                           on='SITE_IDENTIFIER', how='inner')

In [49]:
kin_sub_df_mapped['ENZYME_CLASS_ID'] = enzyme_class_id
kin_sub_df_mapped.rename({
    'PROTEIN_ID':'ENZYME_PROTEIN_ID',
    'MODIFIED_SITE_ID': 'SUBSTRATE_SITE_ID'
}, axis=1, inplace=True)

In [50]:
kin_sub_df_mapped[['ENZYME_PROTEIN_ID', 'SUBSTRATE_SITE_ID', 'ENZYME_CLASS_ID']]

,ENZYME_PROTEIN_ID,SUBSTRATE_SITE_ID,ENZYME_CLASS_ID
0,6176,631378,1
1,6176,631377,1
2,8437,1381010,1
3,8437,1709971,1
4,8437,1673968,1
...,...,...,...
14065,16064,1767262,1
14066,16064,1767264,1
14067,16064,516978,1
14068,16064,323430,1


In [51]:
kin_sub_df_mapped[['ENZYME_PROTEIN_ID', 'SUBSTRATE_SITE_ID', 'ENZYME_CLASS_ID']].to_sql('ENZYME_SUBSTRATE_RELATIONSHIP', conn, if_exists='append', index=False)

14070